# 11 — Lightweight Mel CNN Baseline

Optional PyTorch experiment. Uses patient-disjoint train/validation/test splits; class weights are fit from training labels, early stopping and LR reduction use validation loss, and only training cycles receive noise/time-shift augmentation. If the optional framework is unavailable, install only the optional requirements file.

In [ ]:
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/Users/vs/Sem 5/Respiratory-Disease-Classification"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DL_DIR = PROCESSED_DIR / "deep_learning"

NOTEBOOK_OUTPUT_DIR = DL_DIR / "cnn_baseline"
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42

print("DL_DIR:", DL_DIR)
print("Output directory:", NOTEBOOK_OUTPUT_DIR)

DL_DIR: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning
Output directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/processed/deep_learning/cnn_baseline


In [7]:
manifest_path = DL_DIR / "deep_learning_manifest.csv"

dl_manifest = pd.read_csv(manifest_path)

print("Manifest shape:", dl_manifest.shape)
print("Unique patients:", dl_manifest["patient_id"].nunique())
print("Unique cycles:", len(dl_manifest))

display(dl_manifest.head())

Manifest shape: (6898, 7)
Unique patients: 126
Unique cycles: 6898


,patient_id,recording_id,chest_location,diagnosis,disease_label,official_split,audio_path
0,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
3,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
4,101,1b1,Al,URTI,7,test,/Users/vs/Sem 5/Respiratory-Disease-Classifica...


In [8]:
patient_diagnosis = (
    dl_manifest
    .groupby("patient_id")["diagnosis"]
    .nunique()
)

print(
    "Patients with exactly one diagnosis:",
    (patient_diagnosis == 1).sum()
)

print(
    "Patients with multiple diagnoses:",
    (patient_diagnosis > 1).sum()
)

assert (patient_diagnosis == 1).all()

patient_df = (
    dl_manifest[
        ["patient_id", "diagnosis"]
    ]
    .drop_duplicates()
    .sort_values("patient_id")
    .reset_index(drop=True)
)

print("\nPatient-level dataset shape:", patient_df.shape)

display(patient_df.head())

Patients with exactly one diagnosis: 126
Patients with multiple diagnoses: 0

Patient-level dataset shape: (126, 2)


,patient_id,diagnosis
0,101,URTI
1,102,Healthy
2,103,Asthma
3,104,COPD
4,105,URTI


In [9]:
patient_counts = (
    patient_df["diagnosis"]
    .value_counts()
    .sort_index()
)

display(
    patient_counts
    .rename_axis("diagnosis")
    .reset_index(name="patients")
)

,diagnosis,patients
0,Asthma,1
1,Bronchiectasis,7
2,Bronchiolitis,6
3,COPD,64
4,Healthy,26
5,LRTI,2
6,Pneumonia,6
7,URTI,14


In [11]:
print("Patient-level disease distribution:")
display(
    patient_df["diagnosis"]
    .value_counts()
    .sort_index()
    .rename_axis("diagnosis")
    .reset_index(name="patients")
)

Patient-level disease distribution:


,diagnosis,patients
0,Asthma,1
1,Bronchiectasis,7
2,Bronchiolitis,6
3,COPD,64
4,Healthy,26
5,LRTI,2
6,Pneumonia,6
7,URTI,14
